# SPGC Explorer — Wodehouse stylometry

Интерактивный ноутбук поверх derived-артефактов в `/workspace/spgc/derived/`.
Зависит от того, что `spgc_corpus_stats.py`, `spgc_author_affinity.py` и `ner_filter_affinity.py` уже отработали (см. [SESSION_2026-05-15.md](../SESSION_2026-05-15.md)).

Запускать ячейки сверху вниз. Чтобы переключиться на другого автора или другое слово — менять переменные в ячейках с пометкой ⚙.

## 1. Загрузка артефактов и сводка

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DERIVED = Path('/workspace/spgc/derived')

corpus_meta = json.loads((DERIVED / 'corpus_meta.json').read_text())
author_meta = json.loads((DERIVED / 'wodehouse_affinity_meta.json').read_text())
ner_meta    = json.loads((DERIVED / 'wodehouse_ner_filter_meta.json').read_text())

print('=== baseline corpus ==='); print(json.dumps(corpus_meta, indent=2))
print('\n=== Wodehouse author ===');  print(json.dumps(author_meta, indent=2))
print('\n=== NER filter ===');        print(json.dumps(ner_meta,    indent=2))

## 2. Тoпы по разным критериям

`wodehouse_top200_markers.csv` = clean (без NER-помеченных имён) + фильтр `corpus_count - author_count > 10` (отсев hapax-имён которые NER пропустила).

In [ ]:
df = pd.read_csv(DERIVED / 'wodehouse_top200_markers.csv')
print(df.shape)
df.head(20)

In [ ]:
# Top by raw author frequency (что часто, абсолютно)
df.sort_values('author_count', ascending=False).head(20)

In [ ]:
# Узкие маркеры: часто у автора, редко в корпусе
df[(df['corpus_count'] < 200) & (df['author_count'] > 10)] \
    .sort_values('affinity', ascending=False).head(20)

In [ ]:
# Полный clean CSV (без top-200 фильтра) — всё что не proper noun
df_clean = pd.read_csv(DERIVED / 'wodehouse_affinity_clean.csv')
print(df_clean.shape)
df_clean.head(20)

## 3. Scatter: corpus_count vs author_count (log-log)

Bulk на диагонали = Zipf. Точки сильно выше тренда = слова, непропорционально частые у автора → кандидаты в стилистические маркеры.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
scatter = ax.scatter(df['corpus_count'], df['author_count'],
                     c=df['affinity'], cmap='viridis',
                     s=18, alpha=0.7,
                     norm='log')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('corpus_count'); ax.set_ylabel('author_count')
ax.set_title('Wodehouse top-200 markers vs baseline (color = affinity)')
plt.colorbar(scatter, ax=ax, label='affinity')

# Подписать топ-15 по affinity
top = df.sort_values('affinity', ascending=False).head(15)
for _, r in top.iterrows():
    ax.annotate(r['word'], (r['corpus_count'], r['author_count']),
                fontsize=9, alpha=0.85, xytext=(4, 4), textcoords='offset points')
plt.tight_layout(); plt.show()

## 4. Гистограмма affinity (log10)

Распределение affinity по всему clean-CSV. Хвост вправо = маркеры стиля.

In [ ]:
import numpy as np
vals = df_clean['affinity'].dropna()
vals = vals[vals > 0]
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(np.log10(vals), bins=60, color='steelblue', alpha=0.8)
ax.axvline(0, color='red', lw=1, label='affinity = 1 (neutral)')
ax.set_xlabel('log10(affinity)'); ax.set_ylabel('words'); ax.set_title('Wodehouse clean affinity distribution')
ax.legend(); plt.tight_layout(); plt.show()
print(f'words >1x normal:  {(vals > 1).sum():,}')
print(f'words >10x normal: {(vals > 10).sum():,}')
print(f'words >100x normal:{(vals > 100).sum():,}')

## 5. ⚙ Контексты произвольного слова

Меняй `WORD` и запускай. Если файл уже есть в `contexts/` — читает кеш; иначе зовёт `spgc_contexts.py`.

In [ ]:
import subprocess

WORD   = 'blighter'    # <-- поменять и Shift+Enter
AUTHOR = '^Wodehouse,'
WINDOW = 10
MAX_SAMPLES = 5

out = DERIVED / 'contexts' / f'wodehouse_{WORD}.txt'
if not out.exists():
    cmd = ['python', '/workspace/scripts/spgc_contexts.py',
           '--metadata',   '/workspace/spgc/SPGC-metadata-2018-07-18.csv',
           '--tokens-dir', '/workspace/spgc/SPGC-tokens-2018-07-18',
           '--author',     AUTHOR,
           '--word',       WORD,
           '--window',     str(WINDOW),
           '--max-samples', str(MAX_SAMPLES),
           '--out',        str(out)]
    res = subprocess.run(cmd, capture_output=True, text=True)
    print(res.stdout); print(res.stderr)

print(out.read_text())

## 6. ⚙ Контексты для топ-N маркеров батчем

In [ ]:
import subprocess

TOP_N = 30
src   = DERIVED / 'wodehouse_top200_markers.csv'
out   = DERIVED / 'contexts' / f'wodehouse_top{TOP_N}_batch.txt'

cmd = ['python', '/workspace/scripts/spgc_contexts.py',
       '--metadata',    '/workspace/spgc/SPGC-metadata-2018-07-18.csv',
       '--tokens-dir',  '/workspace/spgc/SPGC-tokens-2018-07-18',
       '--author',      '^Wodehouse,',
       '--words-from',  str(src),
       '--column',      'word',
       '--limit',       str(TOP_N),
       '--window',      '8',
       '--max-samples', '3',
       '--out',         str(out)]
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout[-500:])

print(out.read_text()[:4000], '\n...[truncated]')

## 7. ⚙ Прогнать affinity на другом авторе

Меняй `AUTHOR_REGEX` и `SLUG`. **Важно:** использовать `^Surname,` — иначе ловит middle-name false positives (см. SESSION_2026-05-15 пункт 1).

In [ ]:
import subprocess

AUTHOR_REGEX = '^Doyle,'     # ⚙ Conan Doyle
SLUG         = 'doyle'

cmd = ['python', '/workspace/scripts/spgc_author_affinity.py',
       '--metadata',       '/workspace/spgc/SPGC-metadata-2018-07-18.csv',
       '--counts-dir',     '/workspace/spgc/SPGC-counts-2018-07-18',
       '--corpus-counts',  '/workspace/spgc/derived/corpus_counts.csv',
       '--author',         AUTHOR_REGEX,
       '--slug',           SLUG,
       '--out',            '/workspace/spgc/derived',
       '--min-author-count', '5']
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout[-2000:])
if res.returncode != 0:
    print('STDERR:', res.stderr)

In [ ]:
# Глянуть результат
df_other = pd.read_csv(DERIVED / f'{SLUG}_affinity.csv')
print(df_other.shape)
df_other.head(30)

## 8. Сравнение авторов

Простое пересечение их топ-маркеров. Чем меньше overlap — тем более стилистически различны.

In [ ]:
def top_words(slug, n=200):
    p = DERIVED / f'{slug}_affinity.csv'
    return set(pd.read_csv(p).sort_values('affinity', ascending=False).head(n)['word'])

wode = top_words('wodehouse')
try:
    other = top_words(SLUG)
    print(f'Wodehouse  top-200 unique: {len(wode - other)}')
    print(f'{SLUG.title()} top-200 unique: {len(other - wode)}')
    print(f'Overlap:                    {len(wode & other)}')
    print(f'\nShared words: {sorted(wode & other)[:30]}')
except FileNotFoundError:
    print('Run section 7 first for the other author.')